**ENDICEX — Integración de fuentes económico-financieras e imputación de END HAB**

***1. Antecedentes, construcción de la muestra y procedencia de los datos***

Este cuaderno forma parte del flujo reproducible de preparación de datos de ENDICEX. Su punto de partida es la muestra exploratoria de 111 entidades locales (109 ayuntamientos y las ciudades autónomas de Ceuta y Melilla) construida mediante el procedimiento previo ***ENDICEX_Muestra_111_Cobertura_Provincial_Total.***
En esa etapa anterior se conformó la muestra y se incorporaron los indicadores socioeconómicos procedentes del Instituto Nacional de Estadística (INE), mediante sus interfaces de programación de aplicaciones (API), a partir del Atlas de Distribución de Renta de los Hogares de 2024. Estos datos incluyen las variables de renta y pobreza que se utilizarán posteriormente en el PCA socioeconómico.

La muestra se completó con la información económico-financiera de las cuentas anuales de las entidades, obtenida de fuentes diferenciadas según la disponibilidad y el ámbito territorial:
- ***PICEX*** (Plataforma de rendición de cuentas de las Instituciones de Control Externo): fuente principal de la información económico-financiera de las entidades locales.
- ***IDECAL*** (Sistema de Intercambio de Datos de los Expedientes Contables de la Administración Local): fuente consultada para los municipios de Navarra.
- ***Portal de datos abiertos del Instituto Vasco de Estadística (Eustat)***: fuente consultada para los municipios del País Vasco.

De la integración de estas fuentes procede el archivo de entrada

***ENDICEX_111_con_Datos_Plataforma_sin_imputaciones_para_carga.xlsx***, que conserva la muestra y añade, entre otros datos financieros, los siguientes indicadores:
- END HAB: endeudamiento por habitante.
- GTO HAB: gasto por habitante.
- INV POR HAB: inversión por habitante.
Alcance de este cuaderno: no construye nuevamente la muestra, no consulta directamente las API del INE ni descarga aquí las cuentas anuales de PICEX, IDECAL o Eustat. Carga el libro previamente integrado y prepara los datos financieros para los análisis estadísticos posteriores.

**2. Tratamiento de valores ausentes**
La imputación implementada en este cuaderno se limita a END HAB y únicamente a las observaciones cuya celda contiene la marca literal MEDIANA. El valor se sustituye por la mediana no ponderada del endeudamiento por habitante observado en ayuntamientos del mismo estrato poblacional, redondeada a dos decimales.

El procedimiento no imputa automáticamente GTO HAB ni INV POR HAB, ni sustituye otros valores de deuda que no estén marcados como MEDIANA. Un valor observado de cero se distingue de un dato ausente.

**3. Población donante y comparabilidad institucional**
Se consideran donantes los ayuntamientos con cuentas de 2024 rendidas, con un valor válido y observado de END HAB, y pertenecientes al mismo estrato poblacional que la entidad pendiente de imputación.

Ceuta (51001) y Melilla (52001) se excluyen únicamente como donantes, por su singularidad competencial respecto de los ayuntamientos ordinarios. Permanecen en la muestra de 111 entidades y participan en el PCA financiero y en el K-means posterior cuando reúnen los requisitos de disponibilidad y validez de los indicadores.

**4. Procedimiento reproducible**

1. Cargar ENDICEX_111_con_Datos_Plataforma_sin_imputaciones_para_carga.xlsx.
2. Identificar las observaciones cuyo END HAB contiene la marca MEDIANA.
3. Seleccionar automáticamente la población donante válida dentro de cada estrato poblacional.
4. Calcular la mediana de los valores observados de deuda por habitante y redondearla a dos decimales.
5. Modificar únicamente las celdas autorizadas y generar las hojas de auditoría y control de donantes.
No se establecen de antemano medianas ni tamaños de grupos donantes: se obtienen de los datos presentes en el fichero de entrada de cada ejecución.
5. Trazabilidad, resultados y continuidad analítica
Se conservan los datos del libro original salvo las celdas expresamente imputadas. Para documentar el tratamiento se generan las hojas Auditoria_END_HAB, Control_Donantes y Detalle_Donantes, con identificación de las imputaciones, recuentos de donantes y observaciones de referencia.

***La salida alimenta las siguientes fases de ENDICEX:*** tipificación robusta, PCA socioeconómico y financiero, construcción del espacio tridimensional y K-means. La imputación se realiza antes de estas etapas y no constituye una clasificación fiscalizadora por sí misma.

Los archivos son seudonimizados, no anónimos, dado que codigo_ine permite identificar las entidades. Los resultados tienen finalidad investigadora y exploratoria y no constituyen hallazgos de fiscalización ni valoraciones sobre la gestión de las entidades analizadas.

In [5]:
# -*- coding: utf-8 -*-
"""
ENDICEX | Imputación reproducible de END HAB mediante medianas por estrato.
Archivo de entrada: libro de 111 entidades con identificadores seudonimizados.

La metodología se documenta en el notebook asociado. Ninguna mediana ni
recuento de donantes se introduce de antemano: todos se obtienen del Excel.
"""

### 1. Bibliotecas y configuración
# openpyxl permite modificar los valores sin insertar ni desplazar columnas.
# Los códigos INE e identificadores aleatorios existentes se conservan.
import io
import os
import math
import re
from collections import defaultdict
from decimal import Decimal, ROUND_HALF_UP
from statistics import median

from google.colab import files
from openpyxl import load_workbook


### 2. Criterios metodológicos
# Las ciudades autónomas permanecen en la muestra, pero no se consideran
# donantes porque su régimen competencial difiere del municipal ordinario.
CIUDADES_AUTONOMAS = {'51001', '52001'}

# Los límites del estrato son los empleados en la construcción de la muestra.
# No se fijan valores de END HAB ni cantidades predeterminadas de donantes.
ESTRATOS = (
    '<5.000',
    '5.000–19.999',
    '20.000–99.999',
    '100.000 o más',
)


### 3. Carga del archivo de origen
# Archivo a cargar
ARCHIVO_A_CARGAR = 'ENDICEX_111_con_Datos_Plataforma_sin_imputaciones_para_carga.xlsx'

if os.path.exists(ARCHIVO_A_CARGAR):
    print(f"Cargando archivo local: {ARCHIVO_A_CARGAR}")
    with open(ARCHIVO_A_CARGAR, 'rb') as f:
        contenido = f.read()
    nombre_entrada = ARCHIVO_A_CARGAR
else:
    print('No se encontró el archivo local predeterminado. Por favor, sube el Excel seudonimizado de ENDICEX (sin imputar).')
    archivos = files.upload()
    if len(archivos) != 1:
        raise ValueError('Debes subir exactamente un archivo Excel.')
    nombre_entrada, contenido = next(iter(archivos.items()))

if not nombre_entrada.lower().endswith('.xlsx'):
    raise ValueError('La entrada debe tener formato .xlsx.')

libro = load_workbook(io.BytesIO(contenido), data_only=False)
if 'Muestra_111' not in libro.sheetnames:
    raise ValueError('No existe la hoja Muestra_111.')
hoja = libro['Muestra_111']

# Fotografía de todos los valores originales: control de modificaciones.
hojas_originales = list(libro.worksheets)
valores_originales = {
    (ws.title, celda.coordinate): celda.value
    for ws in hojas_originales
    for fila in ws.iter_rows()
    for celda in fila
}


### 4. Reconocimiento de columnas y controles estructurales
columnas = {
    str(celda.value).strip(): celda.column
    for celda in hoja[1]
    if celda.value is not None
}

campos_obligatorios = {
    'codigo_ine', 'id_aleatorio', 'tipo_entidad',
    'poblacion', 'estrato', 'cuentas 24', 'END HAB',
}
faltantes = campos_obligatorios - set(columnas)
if faltantes:
    raise ValueError(f'Faltan columnas: {sorted(faltantes)}')
if 'municipio' in columnas:
    raise ValueError('Se requiere la versión seudonimizada, sin municipio.')

nombres_auditoria = {
    'Auditoria_END_HAB', 'Control_Donantes',
    'Detalle_Donantes', 'Detalle_91_Donantes',
}
if nombres_auditoria.intersection(libro.sheetnames):
    raise ValueError('El libro contiene hojas de imputación previas.')


### 5. Funciones de limpieza de datos

def normalizar_ine(valor):
    """Representa el código INE con cinco dígitos, preservando ceros."""
    codigo = str(valor).strip()
    if codigo.endswith('.0'):
        codigo = codigo[:-2]
    if not re.fullmatch(r'\d{1,5}', codigo):
        raise ValueError(f'Código INE no válido: {valor!r}')
    return codigo.zfill(5)


def interpretar_end_hab(valor):
    """Devuelve un Decimal no negativo o None para un dato no observable.

    Un valor cero es admisible cuando las cuentas constan como rendidas;
    la ausencia de dato y una deuda observada de cero no son equivalentes.
    """
    if valor is None or isinstance(valor, bool):
        return None
    if isinstance(valor, str):
        texto = re.sub(r'\s+', '', valor).replace('\u00a0', '')
        if not texto or texto.upper() == 'MEDIANA':
            return None
        valor = texto.replace('.', '').replace(',', '.') if ',' in texto else texto
    try:
        numero = Decimal(str(valor))
    except (ValueError, ArithmeticError):
        return None
    return numero if numero.is_finite() and numero >= 0 else None


def cuentas_rendidas_2024(valor):
    """Interpreta el indicador de rendición de cuentas de 2024."""
    return str(valor).strip().casefold() in {'1', '1.0', 'true'}


### 6. Lectura y validación de la muestra
registros = []
for numero_fila in range(2, hoja.max_row + 1):
    valor_codigo = hoja.cell(numero_fila, columnas['codigo_ine']).value
    if valor_codigo is None:
        continue
    registro = {
        campo: hoja.cell(numero_fila, indice).value
        for campo, indice in columnas.items()
    }
    registro['codigo_ine'] = normalizar_ine(valor_codigo)
    registro['_fila_excel'] = numero_fila
    registro['_estrato'] = str(registro['estrato']).strip()
    registros.append(registro)

if len(registros) != 111:
    raise ValueError(f'Muestra inesperada: {len(registros)} entidades (deben ser 111).')
if len({r['codigo_ine'] for r in registros}) != len(registros):
    raise ValueError('Se han detectado códigos INE duplicados.')
identificadores = [str(r['id_aleatorio']).strip() for r in registros]
if any(x.lower() in {'', 'none', 'nan'} for x in identificadores):
    raise ValueError('Existen identificadores aleatorios sin informar.')
if len(set(identificadores)) != len(registros):
    raise ValueError('Se han detectado identificadores aleatorios duplicados.')
if not CIUDADES_AUTONOMAS.issubset({r['codigo_ine'] for r in registros}):
    raise ValueError('No figuran las dos ciudades autónomas en la muestra.')
for registro in registros:
    if registro['_estrato'] not in ESTRATOS:
        raise ValueError(
            f"Estrato desconocido en {registro['codigo_ine']}: "
            f"{registro['_estrato']!r}"
        )


### 7. Selección automática de donantes y pendientes
# Donante = ayuntamiento ordinario, cuentas 2024 rendidas, END HAB
# observado y mismo estrato poblacional que el registro a imputar.
# Solo se imputan las celdas con la marca literal MEDIANA.
donantes = defaultdict(list)
pendientes = []
for registro in registros:
    codigo = registro['codigo_ine']
    estrato = registro['_estrato']
    end_original = registro['END HAB']
    if isinstance(end_original, str) and end_original.strip().upper() == 'MEDIANA':
        pendientes.append(registro)
    elif (
        codigo not in CIUDADES_AUTONOMAS
        and str(registro['tipo_entidad']).strip().casefold() == 'ayuntamiento'
        and cuentas_rendidas_2024(registro['cuentas 24'])
        and interpretar_end_hab(end_original) is not None
    ):
        donantes[estrato].append(registro)

if not pendientes:
    raise ValueError('No se han encontrado celdas END HAB marcadas MEDIANA.')

# Una imputación solo es posible si su estrato tiene valores observados.
para_imputar = {r['_estrato'] for r in pendientes}
for estrato in para_imputar:
    if not donantes[estrato]:
        raise ValueError(f'No hay donantes válidos para el estrato {estrato}.')


### 8. Cálculo de medianas no ponderadas
# No se codifican importes fijos ni se contrastan con cifras prefijadas.
# El redondeo a céntimos se hace con criterio ROUND_HALF_UP.
medianas = {}
for estrato, listado in donantes.items():
    observados = [interpretar_end_hab(r['END HAB']) for r in listado]
    medianas[estrato] = median(observados).quantize(
        Decimal('0.01'), rounding=ROUND_HALF_UP
    )


### 9. Imputaciones y hojas de trazabilidad
celdas_autorizadas = set()
hoja_auditoria = libro.create_sheet('Auditoria_END_HAB')
hoja_auditoria.append([
    'codigo_ine', 'id_aleatorio', 'estrato', 'END HAB original',
    'END HAB imputado', 'N donantes', 'Método',
])
for registro in pendientes:
    celda = hoja.cell(registro['_fila_excel'], columnas['END HAB'])
    celdas_autorizadas.add(('Muestra_111', celda.coordinate))
    estrato = registro['_estrato']
    resultado = float(medianas[estrato])
    celda.value = resultado
    celda.number_format = '#,##0.00'
    hoja_auditoria.append([
        registro['codigo_ine'], registro['id_aleatorio'], estrato,
        'MEDIANA', resultado, len(donantes[estrato]),
        'Mediana no ponderada: ayuntamientos con cuentas 2024 rendidas; '
        'excluidas Ceuta y Melilla como donantes',
    ])

hoja_control = libro.create_sheet('Control_Donantes')
hoja_control.append(['Estrato', 'N donantes', 'Mediana END HAB (€/hab.)'])
for estrato in ESTRATOS:
    hoja_control.append([
        estrato,
        len(donantes[estrato]),
        float(medianas[estrato]) if estrato in medianas else None,
    ])

hoja_detalle = libro.create_sheet('Detalle_Donantes')
hoja_detalle.append([
    'codigo_ine', 'id_aleatorio', 'estrato', 'END HAB observado',
])
for estrato in ESTRATOS:
    for registro in sorted(
        donantes[estrato], key=lambda x: interpretar_end_hab(x['END HAB'])
    ):
        hoja_detalle.append([
            registro['codigo_ine'], registro['id_aleatorio'], estrato,
            float(interpretar_end_hab(registro['END HAB'])),
        ])


### 10. Verificación de integridad y exportación
# Las únicas diferencias de valores en las hojas originales deben ser
# exactamente las celdas END HAB que tenían el marcador MEDIANA.
celdas_modificadas = {
    (ws.title, celda.coordinate)
    for ws in hojas_originales
    for fila in ws.iter_rows()
    for celda in fila
    if celda.value != valores_originales[(ws.title, celda.coordinate)]
}
if celdas_modificadas != celdas_autorizadas:
    raise RuntimeError(
        'Diferencias no autorizadas: '
        f'{sorted(celdas_modificadas ^ celdas_autorizadas)}'
    )

nombre_salida = 'ENDICEX_111_Anonimo_END_HAB_Imputado.xlsx'
libro.save(nombre_salida)

print('\nMEDIANAS CALCULADAS')
for estrato in ESTRATOS:
    n = len(donantes[estrato])
    med = medianas.get(estrato)
    print(f'{estrato:<17} | Donantes: {n:>3} | Mediana: {float(med):.2f} €/hab.'
          if med is not None else f'{estrato:<17} | Donantes: {n:>3} | Sin mediana')
print(f'\nImputaciones realizadas: {len(pendientes)}')
print(f'Observaciones no modificadas: {len(registros) - len(pendientes)}')
print(f'Donantes utilizados: {sum(len(v) for v in donantes.values())}')
print(f'Archivo generado: {nombre_salida}')
files.download(nombre_salida)


Cargando archivo local: ENDICEX_111_con_Datos_Plataforma_sin_imputaciones_para_carga.xlsx

MEDIANAS CALCULADAS
<5.000            | Donantes:  41 | Mediana: 256.14 €/hab.
5.000–19.999      | Donantes:  26 | Mediana: 280.66 €/hab.
20.000–99.999     | Donantes:  18 | Mediana: 428.77 €/hab.
100.000 o más     | Donantes:   6 | Mediana: 631.29 €/hab.

Imputaciones realizadas: 8
Observaciones no modificadas: 103
Donantes utilizados: 91
Archivo generado: ENDICEX_111_Anonimo_END_HAB_Imputado.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>